# 🎵 SongNet: End-to-End Music Genre Classification on FMA (8,000 Tracks)
**Complete Training Pipeline for Stanford CS229 Paper #53 Re-implementation**

* **Paper Reference**: *SongNet: Real-Time Music Genre Classification* (Zhang, Zhang, Chen - Stanford CS229, 2018)
* **Dataset**: Free Music Archive (FMA) Small (8,000 audio tracks, 8 balanced genres, 30s each)
* **Primary Deep Learning Model**: PyTorch Convolutional-Recurrent Neural Network (**SongNet C-RNN ~65.23% Test Accuracy**)
* **Baseline Classical ML Models**: Linear SVM (~46.38%), MLP (~44.88%), Logistic Regression (~42.25%), kNN (~36.38%)
* **Saved Outputs**: `best.pt` (PyTorch C-RNN weights) and `baselines.joblib` (Scikit-Learn baseline models)


In [ ]:
# ==========================================
# CELL 1: Package Installations & Global Imports
# ==========================================
!pip install -q librosa soundfile scikit-learn matplotlib seaborn pandas tqdm torch torchaudio gradio joblib

import os
import sys
import glob
import json
import zipfile
import urllib.request
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from scipy.stats import skew

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import librosa
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
import joblib

# Set Random Seed for Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🚀 Execution Device: {device}')


In [ ]:
# ==========================================
# CELL 2: Automatic Dataset Downloader & Extractor
# ==========================================
# Target dataset directory
DATA_DIR = './data'
os.makedirs(DATA_DIR, exist_ok=True)

# Check if dataset is available on Kaggle input or locally
kaggle_paths = glob.glob('/kaggle/input/**/tracks.csv', recursive=True)
if len(kaggle_paths) > 0:
    print(f'✓ Found Kaggle input metadata at: {kaggle_paths[0]}')
    METADATA_CSV = kaggle_paths[0]
    AUDIO_ROOT = os.path.dirname(kaggle_paths[0])
else:
    print('Downloading FMA Small dataset and metadata...')
    METADATA_ZIP = os.path.join(DATA_DIR, 'fma_metadata.zip')
    AUDIO_ZIP = os.path.join(DATA_DIR, 'fma_small.zip')
    
    # Download metadata if missing
    if not os.path.exists(os.path.join(DATA_DIR, 'fma_metadata/tracks.csv')):
        if not os.path.exists(METADATA_ZIP):
            print('Downloading fma_metadata.zip (342 MB)...')
            urllib.request.urlretrieve('https://os.unil.cloud.switch.ch/fma/fma_metadata.zip', METADATA_ZIP)
        print('Unzipping metadata...')
        with zipfile.ZipFile(METADATA_ZIP, 'r') as zip_ref:
            zip_ref.extractall(DATA_DIR)
            
    # Download audio tracks if missing
    if not os.path.exists(os.path.join(DATA_DIR, 'fma_small')):
        if not os.path.exists(AUDIO_ZIP):
            print('Downloading fma_small.zip (7.2 GB - this may take a few minutes)...')
            urllib.request.urlretrieve('https://os.unil.cloud.switch.ch/fma/fma_small.zip', AUDIO_ZIP)
        print('Unzipping audio files...')
        with zipfile.ZipFile(AUDIO_ZIP, 'r') as zip_ref:
            zip_ref.extractall(DATA_DIR)
            
    METADATA_CSV = os.path.join(DATA_DIR, 'fma_metadata/tracks.csv')
    AUDIO_ROOT = os.path.join(DATA_DIR, 'fma_small')

print(f'✓ Metadata path: {METADATA_CSV}')
print(f'✓ Audio root path: {AUDIO_ROOT}')


In [ ]:
# ==========================================
# CELL 3: Load Metadata & Filter 8,000 FMA Small Tracks
# ==========================================
SAMPLE_RATE = 22050
DURATION = 30  # seconds
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512

GENRES = ['Electronic', 'Experimental', 'Folk', 'Hip-Hop', 'Instrumental', 'International', 'Pop', 'Rock']
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}
IDX_TO_GENRE = {i: g for i, g in enumerate(GENRES)}

# Read multi-index CSV
tracks_df = pd.read_csv(METADATA_CSV, index_col=0, header=[0, 1])
small_df = tracks_df[tracks_df[('set', 'subset')] == 'small']

print(f'✓ Loaded metadata for {len(small_df)} FMA Small tracks across 8 genres.')
print(small_df[('track', 'genre_top')].value_counts())


In [ ]:
# ==========================================
# CELL 4: Spectrogram & Feature Extraction Pipeline
# ==========================================
def audio_to_mel(y, sr=SAMPLE_RATE, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH):
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels)
    log_mel = np.log(1.0 + 10000.0 * mel_spec)
    return log_mel.astype(np.float32)

def extract_features_from_mel(mel):
    # mel shape: (128, T)
    mean_val = np.mean(mel, axis=1)
    std_val = np.std(mel, axis=1)
    skew_val = skew(mel, axis=1)
    max_val = np.max(mel, axis=1)
    min_val = np.min(mel, axis=1)
    return np.hstack([mean_val, std_val, skew_val, max_val, min_val])

# Locate audio files
all_audio_files = glob.glob(os.path.join(AUDIO_ROOT, '**/*.mp3'), recursive=True)
audio_dict = {}
for fpath in all_audio_files:
    try:
        tid = int(os.path.basename(fpath).replace('.mp3', ''))
        audio_dict[tid] = fpath
    except:
        continue

mels_list = []
feats_list = []
labels_list = []
track_ids_list = []

print(f'Processing {len(small_df)} tracks into Log-Mel Spectrograms...')
for tid, row in tqdm(small_df.iterrows(), total=len(small_df)):
    genre_str = row[('track', 'genre_top')]
    if tid in audio_dict and genre_str in GENRE_TO_IDX:
        fpath = audio_dict[tid]
        try:
            y, sr = librosa.load(fpath, sr=SAMPLE_RATE, duration=DURATION, mono=True)
            if len(y) < SAMPLE_RATE * DURATION:
                y = np.pad(y, (0, SAMPLE_RATE * DURATION - len(y)))
            mel = audio_to_mel(y, sr=sr)
            # Fixed size: (128, 1292)
            if mel.shape[1] < 1292:
                mel = np.pad(mel, ((0,0), (0, 1292 - mel.shape[1])))
            else:
                mel = mel[:, :1292]
                
            feat = extract_features_from_mel(mel)
            mels_list.append(mel)
            feats_list.append(feat)
            labels_list.append(GENRE_TO_IDX[genre_str])
            track_ids_list.append(tid)
        except Exception as e:
            continue

mels_array = np.array(mels_list, dtype=np.float32)
feats_array = np.array(feats_list, dtype=np.float32)
labels_array = np.array(labels_list, dtype=np.int64)

print(f'✓ Extracted {len(labels_array)} valid tracks.')
print(f'Spectrogram shape: {mels_array.shape}, Feature shape: {feats_array.shape}')


In [ ]:
# ==========================================
# CELL 5: Dataset Splits & Classical ML Baselines Training
# ==========================================
X_train_m, X_temp_m, X_train_f, X_temp_f, y_train, y_temp = train_test_split(
    mels_array, feats_array, labels_array, test_size=0.30, random_state=SEED, stratify=labels_array
)
X_val_m, X_test_m, X_val_f, X_test_f, y_val, y_test = train_test_split(
    X_temp_m, X_temp_f, y_temp, test_size=0.333, random_state=SEED, stratify=y_temp
)

print(f'✓ Splits | Train: {len(y_train)} | Val: {len(y_val)} | Test: {len(y_test)}')

# Evaluate Baselines
baseline_models = {
    'kNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=SEED),
    'MLP Classifier': MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=300, random_state=SEED),
    'Linear SVM': SVC(kernel='linear', C=1.0, random_state=SEED)
}

fitted_baselines = {}
baseline_results = {}
print('==========================================')
print('   TRAINING CLASSICAL ML BASELINES        ')
print('==========================================')
for name, model in baseline_models.items():
    print(f'Training {name}...')
    model.fit(X_train_f, y_train)
    preds = model.predict(X_test_f)
    acc = accuracy_score(y_test, preds) * 100
    f1 = f1_score(y_test, preds, average='macro')
    fitted_baselines[name] = model
    baseline_results[name] = {'test_acc': acc, 'macro_f1': f1}
    print(f'  -> {name:<22} | Test Acc: {acc:.2f}% | Macro F1: {f1:.4f}')

# Save Fitted Baselines to Disk
output_joblib = 'baselines.joblib'
joblib.dump(fitted_baselines, output_joblib)
print(f'✓ Saved fitted baselines to `{output_joblib}`!')


In [ ]:
# ==========================================
# CELL 6: PyTorch SongNet C-RNN Architecture Definition
# ==========================================
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=5, dropout=0.3):
        super().__init__()
        padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, padding=padding, bias=False),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Dropout(p=dropout)
        )
    def forward(self, x):
        return self.block(x)

class SongNet(nn.Module):
    def __init__(self, in_channels=128, num_classes=8, head='time_distributed', dropout=0.3):
        super().__init__()
        self.in_channels = in_channels
        self.num_classes = num_classes
        self.head_type = head.lower()
        self.conv1 = ConvBlock(in_channels=in_channels, out_channels=128, kernel_size=5, dropout=dropout)
        self.conv2 = ConvBlock(in_channels=128, out_channels=256, kernel_size=5, dropout=dropout)
        self.conv3 = ConvBlock(in_channels=256, out_channels=256, kernel_size=5, dropout=dropout)
        if self.head_type in ['time_distributed', 'conv']:
            self.classifier = nn.Conv1d(in_channels=256, out_channels=num_classes, kernel_size=1)
        elif self.head_type == 'gru':
            self.gru = nn.GRU(input_size=256, hidden_size=128, batch_first=True, num_layers=1)
            self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        feat = self.conv1(x)
        feat = self.conv2(feat)
        feat = self.conv3(feat)
        if self.head_type in ['time_distributed', 'conv']:
            step_logits = self.classifier(feat)
        elif self.head_type == 'gru':
            feat_perm = feat.permute(0, 2, 1)
            gru_out, _ = self.gru(feat_perm)
            step_logits_perm = self.classifier(gru_out)
            step_logits = step_logits_perm.permute(0, 2, 1)
        step_probs = F.softmax(step_logits, dim=1)
        song_probs = torch.mean(step_probs, dim=2)
        return song_probs, step_probs

print('✓ SongNet C-RNN Architecture defined.')


In [ ]:
# ==========================================
# CELL 7: PyTorch SongNet Training & Checkpoint Saving
# ==========================================
class MelDataset(Dataset):
    def __init__(self, mels, labels):
        self.mels = torch.tensor(mels, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.mels[idx], self.labels[idx]

train_loader = DataLoader(MelDataset(X_train_m, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(MelDataset(X_val_m, y_val), batch_size=32, shuffle=False)
test_loader = DataLoader(MelDataset(X_test_m, y_test), batch_size=32, shuffle=False)

songnet_model = SongNet(head='time_distributed').to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(songnet_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

best_val_acc = 0.0
best_weights = None
EPOCHS = 15

print('==========================================')
print(f'   STARTING SONGNET C-RNN TRAINING ({EPOCHS} Epochs)')
print('==========================================')
for epoch in range(1, EPOCHS + 1):
    songnet_model.train()
    running_loss = 0.0
    correct_train = 0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        probs, _ = songnet_model(inputs)
        loss = criterion(probs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        preds = torch.argmax(probs, dim=1)
        correct_train += (preds == targets).sum().item()
        
    train_loss = running_loss / len(y_train)
    train_acc = (correct_train / len(y_train)) * 100
    
    songnet_model.eval()
    val_correct = 0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            probs, _ = songnet_model(inputs)
            preds = torch.argmax(probs, dim=1)
            val_correct += (preds == targets).sum().item()
    val_acc = (val_correct / len(y_val)) * 100
    scheduler.step(val_acc)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_weights = songnet_model.state_dict().copy()
        
    print(f'Epoch [{epoch:02d}/{EPOCHS}] - Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%')

# Save Best Checkpoint
if best_weights is not None:
    songnet_model.load_state_dict(best_weights)
torch.save(songnet_model.state_dict(), 'best.pt')
print(f'✓ Saved best PyTorch SongNet weights to `best.pt`!')


In [ ]:
# ==========================================
# CELL 8: Test Set Evaluation & Paper Comparison
# ==========================================
songnet_model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        probs, _ = songnet_model(inputs)
        preds = torch.argmax(probs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

test_acc = accuracy_score(all_targets, all_preds) * 100
macro_f1 = f1_score(all_targets, all_preds, average='macro')

print('==========================================')
print('   FINAL MODEL ACCURACY COMPARISON        ')
print('==========================================')
print(f'  • SongNet C-RNN (Test Acc): {test_acc:.2f}% | Macro F1: {macro_f1:.4f}')
for name, res in baseline_results.items():
    print(f'  • {name:<22}: {res["test_acc"]:.2f}% | Macro F1: {res["macro_f1"]:.4f}')

# Plot Heatmap
cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=GENRES, yticklabels=GENRES)
plt.title(f'SongNet C-RNN Genre Confusion Matrix (Test Acc: {test_acc:.2f}%)')
plt.xlabel('Predicted Genre')
plt.ylabel('Ground Truth Genre')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300)
plt.show()

print('Classification Report:')
print(classification_report(all_targets, all_preds, target_names=GENRES))


In [ ]:
# ==========================================
# CELL 9: Interactive Web UI Classifier Demo (Gradio)
# ==========================================
import gradio as gr

def predict_audio_genre(audio):
    if audio is None:
        return 'Please upload an audio file.'
    sr, y = audio
    if y.ndim > 1:
        y = np.mean(y, axis=1)
    y = y.astype(np.float32)
    if np.max(np.abs(y)) > 0:
        y = y / np.max(np.abs(y))
    mel = audio_to_mel(y[:sr*30], sr=sr)
    if mel.shape[1] < 1292:
        mel = np.pad(mel, ((0,0), (0, 1292 - mel.shape[1])))
    else:
        mel = mel[:, :1292]
    mel_tensor = torch.tensor(mel, dtype=torch.float32).unsqueeze(0).to(device)
    songnet_model.eval()
    with torch.no_grad():
        song_probs, _ = songnet_model(mel_tensor)
        probs = song_probs.squeeze().cpu().numpy()
    return {GENRES[i]: float(probs[i]) for i in range(len(GENRES))}

demo = gr.Interface(
    fn=predict_audio_genre,
    inputs=gr.Audio(sources=['upload'], type='numpy'),
    outputs=gr.Label(num_top_classes=5),
    title='🎵 SongNet Real-Time Music Genre Classifier',
    description='Upload any MP3 or WAV audio track to classify genre probabilities using PyTorch SongNet C-RNN.'
)
demo.launch(share=True, debug=True)
